In [2]:
# read silver_sales 
df_silver= spark.read.format('delta').load('Tables/dbo/silver_sales')
print(f' NO. of rows: {df_silver.count()}')
print(f'Columns : {df_silver.columns}')
display(df_silver.limit(5))

StatementMeta(, 3d6e6d70-62e3-40d6-b3a0-f08189c91711, 4, Finished, Available, Finished, False)

 NO. of rows: 1268
Columns : ['order_id', 'order_date', 'customer_name', 'region', 'product_category', 'Revenue', 'quantity', 'status', 'revenue_USD']


SynapseWidget(Synapse.DataFrame, e2ec2370-9936-4f2d-9591-0db3998152af)

In [4]:
# extract year and  month from order_date 
from pyspark.sql.functions import col,date_format,sum,avg,round,count
df_dated= df_silver.withColumn(
    'year_month', 
    date_format(col('order_date'), 'yyyy-MM')
) 
display(df_dated.select('order_id','order_date','year_month').limit(5))

StatementMeta(, 3d6e6d70-62e3-40d6-b3a0-f08189c91711, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cffdffa1-0b4d-45e4-9533-45f9804bf7b1)

In [6]:
# aggregate to gold layer 
df_gold = df_dated.groupBy('region','year_month')\
   .agg(
         round(sum('revenue'),2).alias('total_revenue'),
         count('order_id').alias('total_orders'),
         round(avg('revenue'),2).alias('avg_order_value')
   )\
   .orderBy('year_month','region')
print(f'Gold table row count: {df_gold.count()}')
display(df_gold)

StatementMeta(, 3d6e6d70-62e3-40d6-b3a0-f08189c91711, 8, Finished, Available, Finished, False)

Gold table row count: 96


SynapseWidget(Synapse.DataFrame, 46ece081-abc4-43d1-ac58-c5b3051e473d)

In [7]:
df_gold.write\
   .format('delta')\
   .mode('overwrite')\
   .saveAsTable('gold_sales_summary')

StatementMeta(, 3d6e6d70-62e3-40d6-b3a0-f08189c91711, 9, Finished, Available, Finished, False)